## 1. Cài thư viện

Chỉ cần chạy cell này một lần trong mỗi môi trường. Nếu vừa nâng cấp SDK đã import, hãy restart kernel rồi chạy tiếp.


In [1]:
%pip install -q --upgrade "openai>=2,<3" "google-genai>=2.31,<3" pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


## 2. Nhập API key

Key được nhập ẩn bằng `getpass`. Nếu đã có biến môi trường `OPENAI_API_KEY` và `GEMINI_API_KEY`, notebook sẽ dùng chúng.

Tạo key tại [OpenAI](https://platform.openai.com/api-keys) và [Google AI Studio](https://aistudio.google.com/apikey).


In [3]:
import os
import time
from getpass import getpass

import pandas as pd
from IPython.display import display
from openai import OpenAI
from google import genai
from google.genai import types

openai_key = (os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
gemini_key = (os.getenv("GEMINI_API_KEY") or getpass("Gemini API key: ")).strip()
if not openai_key or not gemini_key:
    raise ValueError("Cần nhập đủ hai API key.")

openai_client = OpenAI(api_key=openai_key, timeout=60, max_retries=0)
gemini_client = genai.Client(
    api_key=gemini_key,
    http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(attempts=1)),
)

OPENAI_MODEL = "gpt-5.4-mini"
GEMINI_MODEL = "gemini-3.1-flash-lite"
print("Đã tạo hai client. Chưa gọi model.")


OpenAI API key: ··········
Gemini API key: ··········
Đã tạo hai client. Chưa gọi model.


## 3. Chuẩn bị cùng một prompt

Hai bên nhận cùng chuỗi prompt, không dùng lịch sử hội thoại hay tìm kiếm web. Đây là đoạn văn giả định để có thể kiểm chứng đáp án.

Nếu sửa `text` hoặc câu hỏi, hãy sửa luôn đáp án chuẩn và danh sách ý chính ở mục 6 trước khi chạy phép so sánh.


In [4]:
text = """
Trong một thử nghiệm giả định, 60 sinh viên dùng ứng dụng học từ vựng trong 4 tuần.
Điểm kiểm tra trung bình tăng từ 6,5 lên 7,3 trên thang điểm 10.
Nghiên cứu không có nhóm đối chứng, vì vậy chưa thể kết luận ứng dụng là nguyên nhân
làm điểm số tăng.
""".strip()

prompts = {
    "Tóm tắt": f"Tóm tắt đoạn văn sau bằng tiếng Việt trong đúng 2 câu. Không thêm thông tin.\n\n{text}",
    "QA": f"Chỉ dựa vào đoạn văn, trả lời ngắn gọn: Thử nghiệm kéo dài bao lâu?\n\n{text}",
}

for task, prompt in prompts.items():
    print(f"--- {task} ---\n{prompt}\n")


--- Tóm tắt ---
Tóm tắt đoạn văn sau bằng tiếng Việt trong đúng 2 câu. Không thêm thông tin.

Trong một thử nghiệm giả định, 60 sinh viên dùng ứng dụng học từ vựng trong 4 tuần.
Điểm kiểm tra trung bình tăng từ 6,5 lên 7,3 trên thang điểm 10.
Nghiên cứu không có nhóm đối chứng, vì vậy chưa thể kết luận ứng dụng là nguyên nhân
làm điểm số tăng.

--- QA ---
Chỉ dựa vào đoạn văn, trả lời ngắn gọn: Thử nghiệm kéo dài bao lâu?

Trong một thử nghiệm giả định, 60 sinh viên dùng ứng dụng học từ vựng trong 4 tuần.
Điểm kiểm tra trung bình tăng từ 6,5 lên 7,3 trên thang điểm 10.
Nghiên cứu không có nhóm đối chứng, vì vậy chưa thể kết luận ứng dụng là nguyên nhân
làm điểm số tăng.



## 4. Gọi API và đo thời gian

Hàm dưới đây có hai nhánh: OpenAI và Gemini. Giữ temperature = 0 và giới hạn output = 800 token. OpenAI reasoning = none, Gemini thinking = minimal; hai mức này **không hoàn toàn tương đương**.

Thời gian là từ lúc gửi đến lúc nhận toàn bộ phản hồi, có cả độ trễ mạng. Tắt retry và tự gọi tiếp của Gemini để không âm thầm gửi thêm request.

Giá USD / 1 triệu token, **paid Standard text**, kiểm tra ngày 15/10/2026. Ước tính không phải hóa đơn: chưa tính thuế, ưu đãi hay free tier. Đổi model thì cập nhật cả bảng giá. Thiếu usage/giá sẽ để trống, không gán chi phí bằng 0.


In [5]:
# Thứ tự giá: input thường, cached input, output.
PRICES = {
    "gpt-5.4-mini": (0.75, 0.075, 4.50),
    "gemini-3.1-flash-lite": (0.25, 0.025, 1.50),
}

def call_api(provider, prompt):
    started = time.perf_counter()
    if provider == "ChatGPT":
        model = OPENAI_MODEL
        response = openai_client.responses.create(
            model=model, input=prompt, temperature=0, max_output_tokens=800,
            reasoning={"effort": "none"}, store=False, service_tier="default",
        )
        answer = response.output_text or ""
        status = response.status
        usage = response.usage
        input_tokens = getattr(usage, "input_tokens", None)
        # OpenAI output_tokens đã bao gồm reasoning; không cộng lần nữa.
        output_tokens = getattr(usage, "output_tokens", None)
        cached = getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0
    else:
        model = GEMINI_MODEL
        response = gemini_client.models.generate_content(
            model=model, contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0, max_output_tokens=800, automatic_continuation=False,
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
            ),
        )
        answer = response.text or ""
        candidate = response.candidates[0] if response.candidates else None
        reason = getattr(candidate, "finish_reason", None)
        status = "completed" if reason == types.FinishReason.STOP else str(reason)
        usage = response.usage_metadata
        input_tokens = getattr(usage, "prompt_token_count", None)
        visible = getattr(usage, "candidates_token_count", None)
        thoughts = getattr(usage, "thoughts_token_count", 0) or 0
        output_tokens = visible + thoughts if visible is not None else None
        cached = getattr(usage, "cached_content_token_count", 0) or 0

    seconds = time.perf_counter() - started
    if status == "completed" and not answer.strip():
        status = "empty"
    cost = None
    price = PRICES.get(model)
    if price and input_tokens is not None and output_tokens is not None:
        cost = ((input_tokens - cached) * price[0] + cached * price[1]
                + output_tokens * price[2]) / 1_000_000
    return {
        "provider": provider, "model": model, "status": status,
        "seconds": seconds, "input_tokens": input_tokens, "cached_tokens": cached,
        "output_tokens": output_tokens, "estimated_usd": cost, "answer": answer,
    }


## 5. Chạy và xem kết quả

Cell này gọi API thật và có thể phát sinh phí. Chạy lại cell sẽ gọi thêm **4 request**. Lỗi của một bên không ngăn bên còn lại chạy; không coi request lỗi là kết quả nhanh/rẻ.

Kết quả được lưu thêm vào `ket_qua_chatgpt_gemini.csv` trong thư mục đang chạy notebook.


In [6]:
results = []
for index, (task, prompt) in enumerate(prompts.items()):
    # Đổi bên gọi trước ở ví dụ thứ hai.
    providers = ["ChatGPT", "Gemini"] if index == 0 else ["Gemini", "ChatGPT"]
    for provider in providers:
        try:
            row = call_api(provider, prompt)
        except Exception as error:
            # Không in thông báo thô vì có thể chứa thông tin nhạy cảm.
            row = {
                "provider": provider,
                "model": OPENAI_MODEL if provider == "ChatGPT" else GEMINI_MODEL,
                "status": f"error: {type(error).__name__}",
                "seconds": None, "input_tokens": None, "cached_tokens": None, "output_tokens": None,
                "estimated_usd": None, "answer": "",
            }
        row["task"] = task
        row["prompt"] = prompt
        results.append(row)
        print(f"\n--- {task} | {provider} | {row['status']} ---")
        print(row["answer"] or "(Không có câu trả lời hoàn chỉnh)")

df = pd.DataFrame(results)
columns = ["task", "provider", "model", "status", "seconds",
           "input_tokens", "cached_tokens", "output_tokens", "estimated_usd"]
display(df[columns].round({"seconds": 3, "estimated_usd": 8}))
df.to_csv("ket_qua_chatgpt_gemini.csv", index=False, encoding="utf-8-sig")
print("Đã lưu ket_qua_chatgpt_gemini.csv")



--- Tóm tắt | ChatGPT | completed ---
Trong một thử nghiệm giả định, 60 sinh viên dùng ứng dụng học từ vựng trong 4 tuần và điểm kiểm tra trung bình tăng từ 6,5 lên 7,3 trên thang điểm 10. Tuy nhiên, nghiên cứu không có nhóm đối chứng nên chưa thể kết luận ứng dụng là nguyên nhân làm điểm số tăng.

--- Tóm tắt | Gemini | completed ---
Trong một thử nghiệm kéo dài 4 tuần với 60 sinh viên, điểm kiểm tra trung bình đã tăng từ 6,5 lên 7,3 sau khi sử dụng ứng dụng học từ vựng. Tuy nhiên, do thiếu nhóm đối chứng nên chưa thể khẳng định ứng dụng là nguyên nhân trực tiếp dẫn đến sự cải thiện này.

--- QA | Gemini | completed ---
Thử nghiệm kéo dài 4 tuần.

--- QA | ChatGPT | completed ---
4 tuần.


,task,provider,model,status,seconds,input_tokens,cached_tokens,output_tokens,estimated_usd
0,Tóm tắt,ChatGPT,gpt-5.4-mini,completed,3.998,103,0,76,0.000419
1,Tóm tắt,Gemini,gemini-3.1-flash-lite,completed,2.895,100,0,71,0.000131
2,QA,Gemini,gemini-3.1-flash-lite,completed,0.547,100,0,9,0.000039
3,QA,ChatGPT,gpt-5.4-mini,completed,1.122,105,0,7,0.000110


Đã lưu ket_qua_chatgpt_gemini.csv


## 6. Đánh giá độ chính xác và chất lượng

- **QA:** Exact Match (EM) = 100 nếu câu trả lời sau chuẩn hóa khớp một đáp án chuẩn, ngược lại = 0. Giữ dấu tiếng Việt, bỏ dấu câu và gộp khoảng trắng. Chỉ có 1 câu QA nên đây là điểm của **một câu**, không phải accuracy của model trên một tập dữ liệu lớn. Cách diễn đạt đúng chưa có trong đáp án có thể nhận 0; cần đọc lại output.
- **Tóm tắt:** kiểm tra 5 ý bằng regex và tính độ phủ = số ý khớp / 5 × 100. Đây chỉ là **heuristic về độ phủ**, không phải độ chính xác sự kiện: có thể bỏ sót cách diễn đạt đúng hoặc nhận nhầm số liệu sai ngữ cảnh.
- Để đánh giá đúng sự kiện của tóm tắt, đọc nguồn và output, rồi đổi `summary_factual_ok` thành `True` nếu không sai số liệu, không đảo nghĩa và không thêm thông tin không có trong nguồn; `False` nếu có lỗi. Giữ `None` nếu chưa kiểm tra. Đây là đánh giá thủ công của bạn, không do code tự suy ra.
- Phản hồi lỗi/bị cắt không được chấm như một câu trả lời hoàn chỉnh. Trạng thái luôn được giữ lại.

Cell dưới chỉ phân tích, **không gọi thêm API**.


In [7]:
import re
import unicodedata

# Đáp án/ý chính cố định cho đoạn văn ở mục 3.
qa_answers = ["4 tuần", "bốn tuần", "thử nghiệm kéo dài 4 tuần",
              "thử nghiệm kéo dài bốn tuần", "thời gian thử nghiệm là 4 tuần"]
key_facts = {
    "60 sinh viên": r"(60|sáu mươi)\s+sinh viên",
    "4 tuần": r"(4|bốn)\s+tuần",
    "điểm tăng 6,5 → 7,3": r"6[,.]5.{0,60}7[,.]3",
    "không có nhóm đối chứng": r"không có\s+(?:một\s+)?nhóm đối chứng",
    "chưa thể kết luận nhân quả": r"(?:chưa thể|không thể|chưa đủ.{0,20})\s*(?:kết luận|khẳng định|chứng minh|xác định)",
}
# Sau khi đọc output, điền True / False hoặc giữ None nếu chưa review.
summary_factual_ok = {"ChatGPT": None, "Gemini": None}

def normalize(answer):
    value = unicodedata.normalize("NFC", str(answer)).lower()
    return " ".join(re.sub(r"[^\w\s]", " ", value).split())

def score_results(frame):
    scored = frame.copy()
    scores = []
    for _, row in scored.iterrows():
        score = {"qa_em": None, "key_fact_coverage_pct": None,
                 "matched_facts": "", "missing_facts": "", "summary_factual_ok": None}
        if row["status"] == "completed" and str(row["answer"]).strip():
            if row["task"] == "QA":
                score["qa_em"] = 100 * int(normalize(row["answer"]) in {normalize(a) for a in qa_answers})
            elif row["task"] == "Tóm tắt":
                answer = unicodedata.normalize("NFC", str(row["answer"])).lower()
                found = [name for name, pattern in key_facts.items() if re.search(pattern, answer, re.S)]
                score.update(
                    key_fact_coverage_pct=100 * len(found) / len(key_facts),
                    matched_facts="; ".join(found),
                    missing_facts="; ".join(name for name in key_facts if name not in found),
                    summary_factual_ok=summary_factual_ok.get(row["provider"]),
                )
        scores.append(score)
    return pd.concat([scored.reset_index(drop=True), pd.DataFrame(scores)], axis=1)

evaluation_df = score_results(df) if "df" in globals() and not df.empty else pd.DataFrame()
if evaluation_df.empty:
    print("Chưa có kết quả API thật. Chạy mục 5 trước, chưa có điểm để phân tích.")
else:
    display(evaluation_df[["task", "provider", "status", "qa_em",
                           "key_fact_coverage_pct", "summary_factual_ok"]])
    evaluation_df.to_csv("danh_gia_chatgpt_gemini.csv", index=False, encoding="utf-8-sig")


,task,provider,status,qa_em,key_fact_coverage_pct,summary_factual_ok
0,Tóm tắt,ChatGPT,completed,NaN,100.0,None
1,Tóm tắt,Gemini,completed,NaN,80.0,None
2,QA,Gemini,completed,100.0,NaN,None
3,QA,ChatGPT,completed,100.0,NaN,None


## 7. Phân tích kết quả tự động

Cell này tạo **nhận xét bằng lời**, không chỉ hiển thị bảng:

1. QA: model nào khớp đáp án chuẩn, model nào cần xem lại?
2. Tóm tắt: từng model giữ/bỏ sót ý nào theo heuristic? Đã có đánh giá đúng sự kiện thủ công chưa?
3. Với từng tác vụ, bên nào nhanh hơn và chênh bao nhiêu giây?
4. Bên nào có chi phí ước tính thấp hơn, chênh bao nhiêu USD và dùng bao nhiêu token?

Chỉ so tốc độ/chi phí khi **cả hai phản hồi hoàn chỉnh** và có đủ dữ liệu tương ứng. Không coi lỗi 401/429 là nhanh hoặc miễn phí. Báo cáo lưu vào `phan_tich_chatgpt_gemini.md`.


In [8]:
from pathlib import Path
from IPython.display import Markdown

def compare_measure(group, column, label, unit, digits):
    valid = group[(group["status"] == "completed") & group[column].notna()]
    if len(valid) != 2 or set(valid["provider"]) != {"ChatGPT", "Gemini"}:
        return f"- {label}: chưa đủ hai phản hồi hoàn chỉnh và số liệu để so sánh."
    a = float(valid[valid["provider"] == "ChatGPT"].iloc[0][column])
    b = float(valid[valid["provider"] == "Gemini"].iloc[0][column])
    numbers = f"ChatGPT {a:.{digits}f} {unit}; Gemini {b:.{digits}f} {unit}."
    if abs(a - b) < 1e-12:
        return f"- {label}: {numbers} Hai bên bằng nhau trong lần đo này."
    winner = "ChatGPT" if a < b else "Gemini"
    advantage = "nhanh hơn" if column == "seconds" else "có chi phí ước tính thấp hơn"
    return f"- {label}: {numbers} {winner} {advantage} {abs(a-b):.{digits}f} {unit} trong lần đo này."

def build_analysis(frame):
    report = ["# Phân tích kết quả ChatGPT và Gemini", ""]
    if frame.empty:
        return "\n".join(report + [
            "Chưa có kết quả API thật, nên chưa thể kết luận độ chính xác, tốc độ hay chi phí.",
            "Chạy mục 5, sau đó mục 6–7 để tạo báo cáo từ phản hồi thật.",
        ])
    for task in ("Tóm tắt", "QA"):
        group = frame[frame["task"] == task]
        report += [f"## Tác vụ {task}", ""]
        for _, row in group.iterrows():
            name = row["provider"]
            if row["status"] != "completed":
                report.append(f"- {name}: {row['status']}; không có output hoàn chỉnh để đánh giá chất lượng.")
                continue
            if task == "QA":
                verdict = "khớp đáp án chuẩn '4 tuần'" if row["qa_em"] == 100 else "không khớp đáp án chuẩn; cần kiểm tra cách diễn đạt hoặc lỗi trả lời"
                report.append(f"- Độ chính xác QA — {name}: EM = {row['qa_em']:.0f}/100, {verdict}.")
            else:
                found = row["matched_facts"] or "không có ý nào khớp mẫu"
                missing = row["missing_facts"] or "không có ý nào theo bộ mẫu này"
                report.append(f"- Chất lượng tóm tắt — {name}: độ phủ ý chính (heuristic) {row['key_fact_coverage_pct']:.0f}%. Ý khớp: {found}. Ý chưa khớp: {missing}.")
                review = row["summary_factual_ok"]
                if pd.isna(review):
                    report.append(f"  Chưa review đúng sự kiện cho {name}; độ phủ trên không chứng minh bản tóm tắt chính xác.")
                else:
                    verdict = "không phát hiện lỗi sự kiện/thông tin thêm" if bool(review) else "có lỗi sự kiện hoặc thông tin thêm"
                    report.append(f"  Review thủ công của bạn: {verdict}.")
            tokens = (f"input {row['input_tokens']:.0f}, output tính phí {row['output_tokens']:.0f}"
                      if pd.notna(row["input_tokens"]) and pd.notna(row["output_tokens"]) else "usage chưa đầy đủ")
            report.append(f"  Token của {name}: {tokens}.")
        report.append(compare_measure(group, "seconds", "Tốc độ", "giây", 3))
        report.append(compare_measure(group, "estimated_usd", "Chi phí paid Standard ước tính", "USD", 8))
        report.append("")

    report += ["## Tổng hợp và đánh đổi", ""]
    for name in ("ChatGPT", "Gemini"):
        group = frame[frame["provider"] == name]
        known = group["estimated_usd"].dropna()
        unknown = len(group) - len(known)
        cost = f"{known.sum():.8f} USD" if len(known) else "chưa xác định"
        report.append(f"- {name}: {sum(group['status'] == 'completed')}/{len(group)} phản hồi hoàn chỉnh; phần chi phí đã biết {cost}; {unknown} request chưa rõ chi phí.")
    report += [
        "",
        "Model nhanh/rẻ hơn chưa chắc có chất lượng tốt hơn. Đối chiếu EM của QA, ý bị bỏ sót và review sự kiện của tóm tắt trước khi chọn.",
        "Mỗi model chỉ chạy một lần trên một ví dụ/tác vụ: không có cơ sở xếp hạng chung hay suy ra tốc độ trung bình ổn định. Mạng, kết nối đầu tiên, độ dài output và thinking có thể ảnh hưởng phép đo.",
        "QA EM là chấm khớp đáp án, không bao phủ mọi diễn đạt đúng. Độ phủ regex không kiểm tra đầy đủ ngữ nghĩa; cần đọc output để phát hiện thêm/bịa thông tin.",
        "Chi phí = ((input - cached) × giá input + cached × giá cached + output tính phí × giá output) / 1.000.000.",
        "Giá text paid Standard kiểm tra 15/10/2026, không phải hóa đơn sau free tier/credits/thuế. Nếu có request chưa rõ giá/usage, tổng phần đã biết không phải tổng chi phí toàn bộ.",
        "OpenAI output_tokens đã gồm reasoning; Gemini output tính phí gồm candidates + thoughts. Không so token/giây giữa hai tokenizer như cùng một đơn vị.",
    ]
    return "\n".join(report)

analysis_text = build_analysis(evaluation_df)
display(Markdown(analysis_text))
Path("phan_tich_chatgpt_gemini.md").write_text(analysis_text, encoding="utf-8")
print("Đã lưu phan_tich_chatgpt_gemini.md. Cell này không gọi API.")


# Phân tích kết quả ChatGPT và Gemini

## Tác vụ Tóm tắt

- Chất lượng tóm tắt — ChatGPT: độ phủ ý chính (heuristic) 100%. Ý khớp: 60 sinh viên; 4 tuần; điểm tăng 6,5 → 7,3; không có nhóm đối chứng; chưa thể kết luận nhân quả. Ý chưa khớp: không có ý nào theo bộ mẫu này.
  Chưa review đúng sự kiện cho ChatGPT; độ phủ trên không chứng minh bản tóm tắt chính xác.
  Token của ChatGPT: input 103, output tính phí 76.
- Chất lượng tóm tắt — Gemini: độ phủ ý chính (heuristic) 80%. Ý khớp: 60 sinh viên; 4 tuần; điểm tăng 6,5 → 7,3; chưa thể kết luận nhân quả. Ý chưa khớp: không có nhóm đối chứng.
  Chưa review đúng sự kiện cho Gemini; độ phủ trên không chứng minh bản tóm tắt chính xác.
  Token của Gemini: input 100, output tính phí 71.
- Tốc độ: ChatGPT 3.998 giây; Gemini 2.895 giây. Gemini nhanh hơn 1.103 giây trong lần đo này.
- Chi phí paid Standard ước tính: ChatGPT 0.00041925 USD; Gemini 0.00013150 USD. Gemini có chi phí ước tính thấp hơn 0.00028775 USD trong lần đo này.

## Tác vụ QA

- Độ chính xác QA — Gemini: EM = 100/100, khớp đáp án chuẩn '4 tuần'.
  Token của Gemini: input 100, output tính phí 9.
- Độ chính xác QA — ChatGPT: EM = 100/100, khớp đáp án chuẩn '4 tuần'.
  Token của ChatGPT: input 105, output tính phí 7.
- Tốc độ: ChatGPT 1.122 giây; Gemini 0.547 giây. Gemini nhanh hơn 0.574 giây trong lần đo này.
- Chi phí paid Standard ước tính: ChatGPT 0.00011025 USD; Gemini 0.00003850 USD. Gemini có chi phí ước tính thấp hơn 0.00007175 USD trong lần đo này.

## Tổng hợp và đánh đổi

- ChatGPT: 2/2 phản hồi hoàn chỉnh; phần chi phí đã biết 0.00052950 USD; 0 request chưa rõ chi phí.
- Gemini: 2/2 phản hồi hoàn chỉnh; phần chi phí đã biết 0.00017000 USD; 0 request chưa rõ chi phí.

Model nhanh/rẻ hơn chưa chắc có chất lượng tốt hơn. Đối chiếu EM của QA, ý bị bỏ sót và review sự kiện của tóm tắt trước khi chọn.
Mỗi model chỉ chạy một lần trên một ví dụ/tác vụ: không có cơ sở xếp hạng chung hay suy ra tốc độ trung bình ổn định. Mạng, kết nối đầu tiên, độ dài output và thinking có thể ảnh hưởng phép đo.
QA EM là chấm khớp đáp án, không bao phủ mọi diễn đạt đúng. Độ phủ regex không kiểm tra đầy đủ ngữ nghĩa; cần đọc output để phát hiện thêm/bịa thông tin.
Chi phí = ((input - cached) × giá input + cached × giá cached + output tính phí × giá output) / 1.000.000.
Giá text paid Standard kiểm tra 15/10/2026, không phải hóa đơn sau free tier/credits/thuế. Nếu có request chưa rõ giá/usage, tổng phần đã biết không phải tổng chi phí toàn bộ.
OpenAI output_tokens đã gồm reasoning; Gemini output tính phí gồm candidates + thoughts. Không so token/giây giữa hai tokenizer như cùng một đơn vị.

Đã lưu phan_tich_chatgpt_gemini.md. Cell này không gọi API.
